# AI Text Detector — BERT Fine-tuning
Run this notebook in **Google Colab** with a GPU runtime:  
`Runtime → Change runtime type → T4 GPU`

## 1. Install dependencies

In [20]:
!pip install -q transformers scikit-learn pandas torch rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.2 MB/s eta 0:00:00


## 2. Mount Google Drive & load dataset
1. Upload your `dataset.csv` to your Google Drive (e.g. into a folder called `textdetector`)
2. Run this cell — it will ask you to sign in and grant access

In [6]:
from google.colab import drive
drive.mount("/content/drive")

# Update this path to wherever you put arxiv_pairs_1000 copy.csv in your Drive
DATASET_PATH = "/content/drive/MyDrive/textdetector/arxiv_pairs_1000 copy.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Imports & GPU check

In [54]:
import pandas as pd
import torch
from torch.optim import AdamW
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from rapidfuzz import fuzz
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import RobertaTokenizer, RobertaForSequenceClassification

import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## 4. Load & preprocess data

In [42]:
def convert_to_classification(df):
    human_df = pd.DataFrame({
        "text": df["real_abstract"],
        "label": 0
    })

    ai_df = pd.DataFrame({
        "text": df["ai_abstract"],
        "label": 1
    })
    return pd.concat([human_df, ai_df])

def remove_near_duplicates(df, threshold=85):
    texts = df['text'].tolist()
    keep_idx = []
    for i, t1 in enumerate(texts):
        duplicate = False
        for j in keep_idx:
            t2 = texts[j]
            if fuzz.token_sort_ratio(t1, t2) > threshold:
                duplicate = True
                break
        if not duplicate:
            keep_idx.append(i)
    return df.iloc[keep_idx].reset_index(drop=True)


def preprocess(df):
    df = convert_to_classification(df)
    df = df.dropna(subset=["text"])
    df = remove_near_duplicates(df, 85)

    print("Dataset size:", len(df))
    print(df["label"].value_counts())

    return df


def split(dataframe):
    train_df, temp_df = train_test_split(
        dataframe, test_size=0.2, random_state=42, stratify=dataframe["label"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"]
    )
    return train_df, val_df, test_df

df = pd.read_csv(DATASET_PATH)
df = preprocess(df)
train_df, val_df, test_df = split(df)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Dataset size: 1773
label
0    888
1    885
Name: count, dtype: int64
Train: 1418 | Val: 177 | Test: 178


## 5. Dataset class & DataLoaders

In [43]:
class EssayDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.texts     = dataframe["text"].tolist()
        self.labels    = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long)
        }

tokenizer    = BertTokenizer.from_pretrained("bert-base-uncased")
train_loader = DataLoader(EssayDataset(train_df, tokenizer), batch_size=16, shuffle=True)
val_loader   = DataLoader(EssayDataset(val_df,   tokenizer), batch_size=16, shuffle=False)
test_loader  = DataLoader(EssayDataset(test_df,  tokenizer), batch_size=16, shuffle=False)
print("DataLoaders ready.")

DataLoaders ready.


## 6. Training loop

In [49]:

def train_model(model, train_loader, val_loader, optimizer, device, epochs=3):
    history = {"train_loss": [], "val_acc": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        # training
        model.train()
        total_train_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            outputs.loss.backward()
            clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_train_loss += outputs.loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # validation
        model.eval()
        total_val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_val_loss += outputs.loss.item()

                preds = outputs.logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_acc = correct / total

        # history
        history["train_loss"].append(avg_train_loss)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch}/{epochs} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    return history

## 7. Evaluation loop

In [59]:
def evaluate_model(model, test_loader, device, return_probs=True):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds = logits.argmax(dim=1)
            probs = torch.softmax(logits, dim=1)[:,1]

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    # getting metrics
    accuracy = sum([p==l for p,l in zip(all_preds, all_labels)]) / len(all_labels)
    class_report = classification_report(all_labels, all_preds, digits=4)
    conf_matrix = confusion_matrix(all_labels, all_preds)
    roc_auc = roc_auc_score(all_labels, all_probs) if return_probs else None

    print(f"Test Accuracy: {accuracy:.4f}")
    print("Classification Report:\n", class_report)
    print("Confusion Matrix:\n", conf_matrix)
    if roc_auc is not None:
        print(f"AUC-ROC: {roc_auc:.4f}")


# Train and test BERT model

In [55]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
optimizer = AdamW(bert_model.parameters(), lr=2e-5)

train_model(bert_model, train_loader, val_loader, optimizer, device, epochs=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'train_loss': [0.5765728453237019, 0.2796150167169196, 0.1948148647291858], 'val_acc': [0.9152542372881356, 0.8700564971751412, 0.9548022598870056], 'val_loss': [0.23705787832538286, 0.38728545621658367, 0.1393491654501607]}


In [60]:
print(evaluate_model(bert_model, test_loader, device, return_probs=True))

Test Accuracy: 0.9438
Classification Report:
               precision    recall  f1-score   support

           0     0.9341    0.9551    0.9444        89
           1     0.9540    0.9326    0.9432        89

    accuracy                         0.9438       178
   macro avg     0.9440    0.9438    0.9438       178
weighted avg     0.9440    0.9438    0.9438       178

Confusion Matrix:
 [[85  4]
 [ 6 83]]
AUC-ROC: 0.9880
None


# 9. Train and test RoBERTa model

In [62]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta_model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)
optimizer = torch.optim.AdamW(roberta_model.parameters(), lr=2e-5)

train_model(roberta_model, train_loader, val_loader, optimizer, device, epochs=3)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'train_loss': [0.6754306427548441, 0.613082919227943, 0.5235930006490664],
 'val_acc': [0.6666666666666666, 0.7966101694915254, 0.751412429378531],
 'val_loss': [0.66254789630572, 0.4936518693963687, 0.4689442068338394]}

In [63]:
print(evaluate_model(roberta_model, test_loader, device, return_probs=True))

Test Accuracy: 0.7921
Classification Report:
               precision    recall  f1-score   support

           0     0.7281    0.9326    0.8177        89
           1     0.9062    0.6517    0.7582        89

    accuracy                         0.7921       178
   macro avg     0.8172    0.7921    0.7880       178
weighted avg     0.8172    0.7921    0.7880       178

Confusion Matrix:
 [[83  6]
 [31 58]]
AUC-ROC: 0.8827
None


## 9. Save the model (optional)
This saves to Colab's temporary storage. Download it before the session ends.

In [ ]:
bert_model.save_pretrained("bert_textdetector")
tokenizer.save_pretrained("bert_textdetector")
print("Model saved to ./bert_textdetector/")

# Download as a zip
import shutil
shutil.make_archive("bert_textdetector", "zip", "bert_textdetector")
files.download("bert_textdetector.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./bert_textdetector/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>